# Evaluating Agents with NeMo Evaluator

By now, you've built an agent and learned how to give it tools and memory. That's a huge achievement! But as any engineer will tell you, building is only half the battle. The other half is answering the question: "Does it actually work well?"

In this notebook, we are going to dive deeper into agent evaluation. We won't just eyeball the answers; we will use a systematic approach called **"LLM-as-a-Judge"** using specialized tooling from **NVIDIA NeMo Evaluator**.

### What is LLM-as-a-Judge?
It's exactly what it sounds like. We use a powerful, highly capable Large Language Model (the "Judge") to review the conversation between a User and your Agent. The Judge scores the Agent's responses based on criteria we define, such as:
- **Helpfulness**: Did it solve the user's problem?
- **Correctness**: Was the information accurate?
- **Coherence**: Did the response make sense?

Let's get started by setting up our environment.

## 1. Setup and Secrets

First, we need to load our secrets. You should have already configured your API keys in the previous steps. 

Be sure to set your ``NGC_CLI_ORG`` as well. You can find this under your email in the profile dropdown on the upper-right after you login to [NVIDIA NGC](https://catalog.ngc.nvidia.com/). Be sure it's the same org from which you generated your NVIDIA API Key that you put in the Secrets Manager earlier! 

In [ ]:
# Load your NGC API KEY here. You set this in the Secrets Manager in a previous step.
from dotenv import load_dotenv
_ = load_dotenv("../variables.env")
_ = load_dotenv("../secrets.env")

# Set your NGC org here. What org did you generate your NVIDIA key under?
import os
os.environ['NGC_CLI_ORG'] = "TODO_YOUR_NGC_ORG_HERE"

## 2. Setting up the Infrastructure

NeMo Evaluator is a microservice. This means it runs as its own independent server that we can send requests to. To run it, we need to:
1.  Install the **NGC CLI** (NVIDIA GPU Cloud Command Line Interface) to download the service.
2.  Authenticate with the NVIDIA container registry.
3.  Download and start the service using Docker.

### Install NGC CLI
We'll download and install the CLI tool directly into our environment.

In [ ]:
%%capture
!sudo apt-get install unzip
!cd $HOME && wget --content-disposition https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/versions/4.8.2/files/ngccli_linux.zip -O ngccli_linux.zip && unzip ngccli_linux.zip && chmod u+x ngc-cli/ngc && echo "export PATH=\"$PATH:$(pwd)/ngc-cli\"" >> ~/.bash_profile && source ~/.bash_profile && cd -

### Authenticate Docker
Next, we log in to the NVIDIA Container Registry so we can pull the specialized Docker images.

In [ ]:
!docker login nvcr.io -u '$oauthtoken' -p $NGC_CLI_API_KEY

### Download NeMo Evaluator
Now we use the NGC CLI to download the `nemo-microservices-quickstart` resource. This contains the Docker Compose files we need.

In [ ]:
!/home/workbench/ngc-cli/ngc registry resource download-version "nvidia/nemo-microservices/nemo-microservices-quickstart:25.10"

### Start the Service
We'll spin up the evaluator service in the background. It might take a minute to fully start up.

In [ ]:
%%capture

os.environ['NEMO_MICROSERVICES_IMAGE_REGISTRY'] = "nvcr.io/nvidia/nemo-microservices"
os.environ['NEMO_MICROSERVICES_IMAGE_TAG'] = "25.10"

!cd nemo-microservices-quickstart_v25.10 && docker compose --profile evaluator up --detach --quiet-pull --wait && cd -

### Network Magic
Since the evaluator is running in a Docker container, we need to figure out the correct IP address to talk to it. The code below handles that network discovery and sets the base URLs for the **Evaluator** and the **Datastore** (where we keep our test data).

In [ ]:
import os, subprocess
import socket

# Try to get the gateway from /proc/net/route (Linux only)
with open('/proc/net/route') as f:
    for line in f:
        fields = line.strip().split()
        if fields[1] == '00000000':  # Default route
            gateway_hex = fields[2]
            # Convert hex to IP
            gateway_ip = socket.inet_ntoa(bytes.fromhex(gateway_hex)[::-1])
            print(f"Gateway IP: {gateway_ip}")
            break

os.environ["EVALUATOR_BASE_URL"] = f"http://{gateway_ip}:8080"
os.environ["NEMO_DATASTORE_URL"] = f"http://{gateway_ip}:3000"

# Namespace and dataset name used for hf:// URL
NAMESPACE = "default"
DATASET_NAME = "my-repo"

## 3. Preparing the Data

To evaluate an agent, we need data! 

We will use the **HelpSteer2** dataset. It contains pairs of Prompts (user questions) and Responses (model answers). In a real scenario, these "Responses" would be generated by *your* agent. Here, we'll use the pre-existing responses in the dataset to test our Judge.

We'll select the first 30 rows and save them as a JSONL (JSON Lines) file.

In [ ]:
import requests
import pandas as pd

# Download the HelpSteer2 dataset from Hugging Face
df = pd.read_json("hf://datasets/nvidia/HelpSteer2/train.jsonl.gz", lines=True)

# Extract only the prompt and response columns for evaluation
df = df[["prompt", "response"]].head(30)

# Save to a local file
file_name = "helpsteer2.jsonl"
df.to_json(file_name, orient="records", lines=True)

print(f"Dataset prepared with {len(df)} samples")
print(f"Sample data:")
print(df.head())

### Uploading to the Datastore
The NeMo Evaluator needs access to this file. It runs a local service that mimics the Hugging Face API (neat, right?). We'll upload our `helpsteer2.jsonl` file to this local datastore so the Judge can access it.

In [ ]:
import os
from huggingface_hub import HfApi

HF_ENDPOINT = f"{os.environ['NEMO_DATASTORE_URL']}/v1/hf"

hf_api = HfApi(endpoint=HF_ENDPOINT, token=os.environ["HF_TOKEN"])
repo_id = f"{NAMESPACE}/{DATASET_NAME}"

# Create the dataset repo if it doesn't exist
hf_api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

# Upload the file
result = hf_api.upload_file(
    path_or_fileobj=file_name,
    path_in_repo=file_name,
    repo_id=repo_id,
    repo_type="dataset",
    revision="main",
    commit_message=f"Eval dataset in {repo_id}"
)

print(f"Dataset uploaded: {result}")

## 4. Configuring the Judge

This is the most critical part. We need to tell NeMo Evaluator **how** to judge the responses. We do this with a configuration dictionary.

### The "Prompt Template"
Look at the `template` section in the config below. We are giving the Judge model a persona: 
> *"You are an expert evaluator... Your task is to assess responses..."*

We then ask it to calculate specific metrics:
1.  **Helpfulness** (0-4)
2.  **Correctness** (0-4)
3.  **Coherence** (0-4)
4.  **Complexity** (0-4)
5.  **Verbosity** (0-4)

We also use **Regular Expressions (Regex)** in the `scores` section to teach the evaluator how to read the Judge's output and extract the numerical scores.

Finally, we initialize the `NeMoMicroservices` client and run the evaluation job.

In [ ]:
import os
from nemo_microservices import NeMoMicroservices

client = NeMoMicroservices(
    base_url=os.environ["EVALUATOR_BASE_URL"],
    timeout=600.0  # 10 minutes instead of default 60 seconds
)

# Model endpoint settings for the judge
MODEL_BASE_URL = "https://integrate.api.nvidia.com/v1"  # Add the url of your endpoint; it can be any model endpoint, such as an OpenAI endpoint, or a NIM endpoint.
MODEL_ID = "nvidia/llama-3.3-nemotron-super-49b-v1.5"                         # replace as needed

files_url = f"hf://datasets/{NAMESPACE}/{DATASET_NAME}"

# Inline config mirrors the developer notebook
config = {
    "type": "custom",
    "name": "my-config",
    "namespace": NAMESPACE,
    "params": {
        "limit_samples": 1,  # Only evaluate top row for testing
        "parallelism": 1,    # Reduce concurrent requests if hitting rate limits
        "request_timeout": 120,  # 2 minutes per request
        "max_retries": 3     # Retry failed requests
    },
    "tasks": {
        "my-task": {
            "type": "data",
            "metrics": {
                "my_eval": {
                    "type": "llm-judge",
                    "params": {
                        "model": {
                            "api_endpoint": {
                                "url": MODEL_BASE_URL,
                                "model_id": MODEL_ID,
                                "format": "openai",
                                "api_key": os.environ["NVIDIA_NIM_API_KEY"]
                            },
                            "prompt": {
                                "inference_params": {
                                    "temperature": 0.1,
                                    "max_tokens": 1024,
                                    "max_retries": 10,
                                    "request_timeout": 120  # THIS IS THE KEY - timeout for judge API calls
                                }
                            }
                        },
                        "template": {
                            "messages": [
                                {"role": "system", "content": "You are an expert evaluator for answers to user queries. Your task is to assess responses to user queries based on helpfulness, relevance, accuracy, and clarity."},
                                {"role": "user", "content": "Calculate the following metrics for the response: User Query: {{item.prompt}} Model Response: {{item.response}} Metrics: 1. Helpfulness (0-4): How well does the response help the user? 2. Correctness (0-4): Is the information correct? 3. Coherence (0-4): Is the response logically consistent and well-structured? 4. Complexity (0-4): How sophisticated is the response? 5. Verbosity (0-4): Is the response appropriately detailed? Instructions: Assign a score from 0 (poor) to 4 (excellent) for each metric. Respond in JSON format only: { \"helpfulness\": ..., \"correctness\": ..., \"coherence\": ..., \"complexity\": ..., \"verbosity\": ... }"}
                            ]
                        },
                        "scores": {
                            "helpfulness": {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"helpfulness\": *(\\d+)"}},
                            "correctness": {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"correctness\": *(\\d+)"}},
                            "coherence":   {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"coherence\": *(\\d+)"}},
                            "complexity":  {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"complexity\": *(\\d+)"}},
                            "verbosity":   {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"verbosity\": *(\\d+)"}}
                        }
                    }
                }
            },
            "dataset": {"files_url": files_url}
        }
    }
}

target = {"type": "dataset", "dataset": {"files_url": files_url}}

job = client.evaluation.live(
    namespace=NAMESPACE,
    target=target,
    config=config,
    timeout=600.0  # Also override per-request
)

## 5. Analyzing Results

The evaluation job runs asynchronously. We can check its `status` and `result`. 

The result object contains a breakdown of the scores for the task.

In [ ]:
# Access the results
print(f"Status: {job.status}\n")
print(f"Results: {job.result}\n")
print(f"Scores: {job.result.tasks['my-task'].metrics['my_eval'].scores}")

### Saving Results
It's always good practice to save your results. We'll dump the full JSON output to a file so we can inspect it later or load it into a dashboard.

In [ ]:
import json

# Save results to file
results_dict = {
    "status": job.status,
    "result": job.result.model_dump(mode='json') if hasattr(job.result, 'model_dump') else job.result
}

with open("evaluation_results.json", "w") as f:
    json.dump(results_dict, f, indent=2, default=str)  # default=str as fallback

print("Saved to evaluation_results.json")

# If you want to access specific scores:
if job.result and job.result.tasks:
    for task_name, task_result in job.result.tasks.items():
        print(f"\nTask: {task_name}")
        if task_result.metrics:
            for metric_name, metric_result in task_result.metrics.items():
                print(f"  Metric: {metric_name}")
                if hasattr(metric_result, 'scores'):
                    print(f"  Scores: {metric_result.scores}")

## 6. Scaling Up (Batch Processing)

In production, you might have thousands of conversations to evaluate. To avoid hitting rate limits on the Judge model API and to handle failures gracefully, we process the data in **batches**.

The loop below processes the data 1 sample at a time (for safety in this workshop environment), but you can adjust `batch_size` in a real deployment.

In [ ]:
# Run multiple batches. Limiting to 1 sample per batch due to rate limits. 
batch_size = 1
all_results = []

for batch_num in range(0, 20, batch_size):  # Adjust total samples as needed
    print(f"Processing batch starting at sample {batch_num}...")
    
    # Add limit and offset to config for this batch
    batch_config = config.copy()
    batch_config["params"]["limit_samples"] = batch_size
    batch_config["params"]["offset"] = batch_num
    
    target = {"type": "dataset", "dataset": {"files_url": files_url}}
    
    try:
        response = client.evaluation.live(
            namespace=NAMESPACE,
            target=target,
            config=batch_config,
            timeout=600.0
        )
        
        all_results.append(response.result.model_dump(mode='json'))
        print(f"✓ Batch {batch_num} completed")
        
    except Exception as e:
        print(f"✗ Batch {batch_num} failed: {e}")
        continue

# Save combined results
with open("evaluation_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"Completed {len(all_results)} batches, saved to evaluation_results.json")

## Conclusion

You have now set up a complete evaluation pipeline! 

1.  You downloaded and deployed **NeMo Evaluator**.
2.  You prepared a **test dataset** (HelpSteer2).
3.  You configured an **LLM-as-a-Judge** to score responses on multiple metrics.
4.  You ran the evaluation and saved the results.

As you continue building agents, remember: **You can't improve what you don't measure.** Use this pipeline to benchmark your agent's performance as you tweak its prompts and tools.